In [1]:
import pygame
import math
import random
import time
from queue import PriorityQueue

pygame.init()
pygame.font.init()

WIDTH = 800
ROWS = 40

WIN = pygame.display.set_mode((WIDTH, WIDTH))
pygame.display.set_caption("Dynamic Pathfinding AI")

FONT = pygame.font.SysFont("comicsans",20)

WHITE=(255,255,255)
BLACK=(0,0,0)
RED=(255,0,0)
TURQUOISE=(64,224,208)
YELLOW=(255,255,0)
BLUE=(0,0,255)
GREEN=(0,255,0)
PURPLE=(128,0,128)
GREY=(128,128,128)

class Node:

    def __init__(self,row,col,width,total_rows):
        self.row=row
        self.col=col
        self.x=row*width
        self.y=col*width
        self.color=WHITE
        self.neighbors=[]
        self.width=width
        self.total_rows=total_rows

    def get_pos(self):
        return self.row,self.col

    def is_obstacle(self):
        return self.color==BLACK

    def is_start(self):
        return self.color==RED

    def is_end(self):
        return self.color==TURQUOISE

    def reset(self): self.color=WHITE
    def make_start(self): self.color=RED
    def make_closed(self): self.color=BLUE
    def make_open(self): self.color=YELLOW
    def make_obstacle(self): self.color=BLACK
    def make_end(self): self.color=TURQUOISE
    def make_path(self): self.color=GREEN
    def make_agent(self): self.color=PURPLE

    def draw(self,win):
        pygame.draw.rect(win,self.color,
                         (self.x,self.y,self.width,self.width))

    def update_neighbors(self,grid):

        self.neighbors=[]

        if self.row<self.total_rows-1 and not grid[self.row+1][self.col].is_obstacle():
            self.neighbors.append(grid[self.row+1][self.col])

        if self.row>0 and not grid[self.row-1][self.col].is_obstacle():
            self.neighbors.append(grid[self.row-1][self.col])

        if self.col<self.total_rows-1 and not grid[self.row][self.col+1].is_obstacle():
            self.neighbors.append(grid[self.row][self.col+1])

        if self.col>0 and not grid[self.row][self.col-1].is_obstacle():
            self.neighbors.append(grid[self.row][self.col-1])

C:\Users\Alyan Tariq\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
def h_manhattan(p1,p2):
    return abs(p1[0]-p2[0])+abs(p1[1]-p2[1])

def h_euclidean(p1,p2):
    return math.sqrt((p1[0]-p2[0])**2+(p1[1]-p2[1])**2)


def make_grid(rows,width):
    grid=[]
    gap=width//rows

    for i in range(rows):
        grid.append([])
        for j in range(rows):
            grid[i].append(Node(i,j,gap,rows))

    return grid


def generate_random_map(grid,start,end,density=0.3):
    for row in grid:
        for node in row:
            if node!=start and node!=end:
                if random.random()<density:
                    node.make_obstacle()
                else:
                    node.reset()


def draw_grid(win,rows,width):
    gap=width//rows

    for i in range(rows):
        pygame.draw.line(win,GREY,(0,i*gap),(width,i*gap))
        for j in range(rows):
            pygame.draw.line(win,GREY,(j*gap,0),(j*gap,width))


def draw_metrics(win,algo,heur,visited,cost,time_ms,dynamic_mode,rows):

    pygame.draw.rect(win,BLACK,(0,0,WIDTH,30))

    text=f"{algo} | {heur} | Visited:{visited} | Cost:{cost} | Time:{time_ms:.2f}ms | Dynamic:{dynamic_mode} | Grid:{rows}"

    label=FONT.render(text,1,WHITE)
    win.blit(label,(10,5))


def draw_all(win,grid,rows,width,algo,heur,visited,cost,time_ms,dynamic):

    win.fill(WHITE)

    for row in grid:
        for node in row:
            node.draw(win)

    draw_grid(win,rows,width)
    draw_metrics(win,algo,heur,visited,cost,time_ms,dynamic,rows)

    pygame.display.update()


def reconstruct_path(came_from,current,draw):

    path=[]
    cost=0

    while current in came_from:
        current=came_from[current]
        if not current.is_start():
            current.make_path()
        path.append(current)
        cost+=1
        draw()

    path.reverse()
    return path,cost

In [3]:
def run_search(draw,grid,start,end,algorithm,heuristic):

    start_time=time.time()

    open_set=PriorityQueue()
    open_set.put((0,0,start))

    came_from={}

    g_score={node:float("inf") for row in grid for node in row}
    f_score={node:float("inf") for row in grid for node in row}

    g_score[start]=0
    f_score[start]=heuristic(start.get_pos(),end.get_pos())

    open_hash={start}
    visited=0
    count=0

    while not open_set.empty():

        for event in pygame.event.get():
            if event.type==pygame.QUIT:
                pygame.quit()

        current=open_set.get()[2]
        open_hash.remove(current)

        if current==end:
            exec_time=(time.time()-start_time)*1000
            path,cost=reconstruct_path(came_from,end,draw)
            end.make_end()
            return True,visited,cost,exec_time,path

        for neighbor in current.neighbors:

            temp_g=g_score[current]+1

            if temp_g<g_score[neighbor]:

                came_from[neighbor]=current
                g_score[neighbor]=temp_g

                h=heuristic(neighbor.get_pos(),end.get_pos())

                if algorithm=="A*":
                    f_score[neighbor]=temp_g+h
                else:
                    f_score[neighbor]=h

                if neighbor not in open_hash:
                    count+=1
                    open_set.put((f_score[neighbor],count,neighbor))
                    open_hash.add(neighbor)
                    neighbor.make_open()

        draw()

        if current!=start:
            current.make_closed()
            visited+=1

    return False,visited,0,0,[]


def spawn_dynamic_wall(grid,start,end):

    candidates=[node for row in grid for node in row
                if node.color==WHITE and node!=start and node!=end]

    if candidates and random.random()<0.2:
        wall=random.choice(candidates)
        wall.make_obstacle()
        return wall

    return None

In [4]:
def main(win,width):

    global ROWS

    grid=make_grid(ROWS,width)

    start=None
    end=None

    algo="A*"
    heuristic=h_manhattan
    h_name="Manhattan"

    visited=cost=exec_time=0
    dynamic_mode=False

    run=True

    while run:

        draw_all(win,grid,ROWS,width,
                 algo,h_name,
                 visited,cost,exec_time,
                 dynamic_mode)

        for event in pygame.event.get():

            if event.type==pygame.QUIT:
                run=False

            # mouse placement
            if pygame.mouse.get_pressed()[0]:

                x,y=pygame.mouse.get_pos()

                if y>30:
                    row=x//(width//ROWS)
                    col=y//(width//ROWS)
                    node=grid[row][col]

                    if not start:
                        start=node
                        start.make_start()

                    elif not end:
                        end=node
                        end.make_end()

                    elif node!=start and node!=end:
                        node.make_obstacle()

            elif pygame.mouse.get_pressed()[2]:

                x,y=pygame.mouse.get_pos()

                if y>30:
                    row=x//(width//ROWS)
                    col=y//(width//ROWS)
                    node=grid[row][col]
                    node.reset()

                    if node==start: start=None
                    if node==end: end=None

            if event.type==pygame.KEYDOWN:

                if event.key==pygame.K_a: algo="A*"
                if event.key==pygame.K_g: algo="GBFS"

                if event.key==pygame.K_m:
                    heuristic=h_manhattan
                    h_name="Manhattan"

                if event.key==pygame.K_e:
                    heuristic=h_euclidean
                    h_name="Euclidean"

                if event.key==pygame.K_d:
                    dynamic_mode=not dynamic_mode

                if event.key==pygame.K_r:
                    generate_random_map(grid,start,end,0.3)

                if event.key==pygame.K_c:
                    start=end=None
                    grid=make_grid(ROWS,width)

                if event.key==pygame.K_UP:
                    ROWS+=5
                    grid=make_grid(ROWS,width)

                if event.key==pygame.K_DOWN and ROWS>10:
                    ROWS-=5
                    grid=make_grid(ROWS,width)

                if event.key==pygame.K_SPACE and start and end:

                    for row in grid:
                        for node in row:
                            node.update_neighbors(grid)

                    def redraw():
                        draw_all(win,grid,ROWS,width,
                                 algo,h_name,
                                 visited,cost,exec_time,
                                 dynamic_mode)

                    success,visited,cost,exec_time,path=run_search(
                        redraw,grid,start,end,algo,heuristic)

                    if success:

                        agent=start
                        i=0

                        while i<len(path) and run:

                            pygame.time.delay(100)

                            if agent!=start:
                                agent.make_path()

                            agent=path[i]
                            agent.make_agent()
                            i+=1

                            if dynamic_mode:
                                new_wall=spawn_dynamic_wall(grid,start,end)

                                if new_wall in path[i:]:

                                    for row in grid:
                                        for node in row:
                                            if node.color in (YELLOW,BLUE,GREEN):
                                                node.reset()
                                            node.update_neighbors(grid)

                                    success,_,_,_,path=run_search(
                                        redraw,grid,agent,end,algo,heuristic)

                                    if not success:
                                        print("Agent trapped")
                                        break

                                    i=0

                            redraw()

    pygame.quit()


if __name__=="__main__":
    main(WIN,WIDTH)